# Game 1 - Multivariate Pair Trading (Coffee, G17)

**Strategie**: basket market-neutral via cointégration + optimisation Gurobi  
**Anchor**: KC=F (Coffee C Futures)  
**Univers**: actions liées au café + ETF DBA

Approche inspirée de Yang & Malik (2024). Formation: 2021-2022, trading: 2023-2026.

> Colab: uploader coffee_dataset_G17.csv puis run all.

In [ ]:
# pour Colab: décommenter
# !pip install gurobipy yfinance statsmodels

import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import pearsonr
from statsmodels.tsa.stattools import adfuller
import gurobipy as gp
from gurobipy import GRB

os.makedirs('figures', exist_ok=True)

# style compatible toutes versions matplotlib
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except OSError:
    plt.style.use('seaborn-darkgrid')

pd.set_option('display.float_format', '{:.4f}'.format)
print('imports OK')

## 1. Chargement des données

On charge le CSV avec les prix de clôture (OHLCV daily, 2021-2026).

In [ ]:
# from google.colab import files
# files.upload()  # choisir coffee_dataset_G17.csv

CSV_PATH = 'coffee_dataset_G17.csv'

df_raw = pd.read_csv(CSV_PATH, index_col=0, parse_dates=True)
df_raw.index.name = 'Date'
df_raw = df_raw.sort_index()

print(df_raw.shape)
print(df_raw.index[0].date(), '->', df_raw.index[-1].date())
df_raw.head()

## 2. Nettoyage des données

In [ ]:
# drop colonnes avec >20% NaN
threshold = 0.80
df_clean = df_raw.dropna(thresh=int(threshold * len(df_raw)), axis=1)
print('colonnes gardées:', list(df_clean.columns))

df_clean = df_clean.ffill().dropna()

# outliers extremes sur les returns (|zscore| > 5)
from scipy.stats import zscore as _zscore
for col in df_clean.columns:
    zs = _zscore(df_clean[col].pct_change().dropna())
    bad_idx = df_clean.index[1:][np.abs(zs) > 5]
    df_clean.loc[bad_idx, col] = np.nan
df_clean = df_clean.ffill().dropna()

ANCHOR  = 'KC=F'
PROXIES = [c for c in df_clean.columns if c in ['DBA']]
STOCKS  = [c for c in df_clean.columns if c not in [ANCHOR] + PROXIES]
ALL_ASSETS = [ANCHOR] + PROXIES + STOCKS

print(f'Anchor: {ANCHOR}')
print(f'Stocks ({len(STOCKS)}): {STOCKS}')
print(f'N obs: {len(df_clean)}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))
norm = df_clean / df_clean.iloc[0] * 100

for col in [ANCHOR] + PROXIES:
    axes[0].plot(norm.index, norm[col], label=col, linewidth=2)
axes[0].set_title('Anchor & proxy - prix normalisé (base 100)', fontsize=12)
axes[0].set_ylabel('Index')
axes[0].legend()

for col in STOCKS:
    axes[1].plot(norm.index, norm[col], label=col, linewidth=1.2, alpha=0.85)
axes[1].set_title('Univers equity café - prix normalisé (base 100)', fontsize=12)
axes[1].set_ylabel('Index')
axes[1].legend(fontsize=8, ncol=3)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())

plt.tight_layout()
plt.savefig('figures/fig_prices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = df_clean.pct_change().dropna().corr()
im = ax.imshow(corr_matrix, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(corr_matrix.columns, fontsize=9)
for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix)):
        ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
ax.set_title('Matrice de corrélation des rendements journaliers', fontsize=12)
plt.tight_layout()
plt.savefig('figures/fig_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Test de cointégration

Test Engle-Granger (ADF sur résidus OLS) de chaque action contre KC=F.  
Seuil p-value < 0.10. On garde les 5 meilleures.

In [ ]:
def engle_granger_test(y, x, significance=0.10):
    """OLS puis ADF sur les résidus. Retourne (pvalue, is_coint, hedge_ratio)."""
    # aligner les index
    common = y.index.intersection(x.index)
    y, x = y.loc[common], x.loc[common]
    if len(y) < 30:
        return 1.0, False, 0.0
    X = np.column_stack([np.ones(len(x)), x.values])
    beta = np.linalg.lstsq(X, y.values, rcond=None)[0]
    resid = y.values - X @ beta
    adf_stat, pvalue, *_ = adfuller(resid, regression='c')
    return pvalue, pvalue < significance, float(beta[1])


log_prices = np.log(df_clean)
anchor_log = log_prices[ANCHOR]

# calcul unique (evite double appel adfuller en cellule 9)
results = []
_hedge_cache = {}
for stock in STOCKS + PROXIES:
    stock_log = log_prices[stock]
    rets_a = anchor_log.diff().dropna()
    rets_s = stock_log.diff().dropna()
    common = rets_a.index.intersection(rets_s.index)
    from scipy.stats import pearsonr as _pr
    corr, _ = _pr(rets_a.loc[common], rets_s.loc[common])
    pval, is_coint, hedge = engle_granger_test(stock_log, anchor_log)
    _hedge_cache[stock] = hedge
    results.append({
        'Asset'       : stock,
        'Correlation' : round(corr, 4),
        'EG p-value'  : round(pval, 4),
        'Cointegrated': is_coint,
        'Hedge Ratio' : round(hedge, 4)
    })

coint_df = pd.DataFrame(results).sort_values('EG p-value')
coint_df

In [ ]:
N_SELECT = 5
selected = coint_df.sort_values('EG p-value').head(N_SELECT)['Asset'].tolist()
print(f'Basket: {selected}')

# reutilise le cache — pas de second appel adfuller
hedge_ratios = {s: _hedge_cache[s] for s in selected}
for s, h in hedge_ratios.items():
    print(f'  {s}: beta = {h:.4f}')

## 4. Optimisation Gurobi

On cherche les poids w qui minimisent la variance sous contrainte de neutralité marché :

$$\min_{w} w^\top \Sigma w \quad \text{s.t.} \quad \sum_i \beta_i w_i = 0, \quad \sum_i w_i = 1$$

La contrainte sum(beta * w) = 0 annule le beta net sur les futures café.

In [ ]:
def optimize_basket(selected_stocks, log_prices, hedge_ratios, allow_short=True, verbose=False):
    n = len(selected_stocks)
    rets = log_prices[selected_stocks].diff().dropna()
    Sigma = rets.cov().values
    betas = np.array([hedge_ratios[s] for s in selected_stocks])

    model = gp.Model('MarketNeutralBasket')
    model.Params.OutputFlag = int(verbose)
    model.Params.TimeLimit = 60

    lo = -1.0 if allow_short else 0.0
    w = model.addVars(n, lb=lo, ub=1.0, name='w')

    # minimiser la variance
    variance = gp.quicksum(Sigma[i, j] * w[i] * w[j] for i in range(n) for j in range(n))
    model.setObjective(variance, GRB.MINIMIZE)

    model.addConstr(gp.quicksum(w[i] for i in range(n)) == 1, 'fully_invested')
    model.addConstr(gp.quicksum(betas[i] * w[i] for i in range(n)) == 0, 'market_neutral')

    model.optimize()

    if model.status == GRB.OPTIMAL:
        return pd.Series([w[i].X for i in range(n)], index=selected_stocks)
    else:
        raise RuntimeError(f'Gurobi status: {model.status}')


FORMATION_END = '2022-12-31'
TRADING_START = '2023-01-01'

log_formation = log_prices.loc[:FORMATION_END]
log_trading   = log_prices.loc[TRADING_START:]
prices_trading = df_clean.loc[TRADING_START:]

print(f'Formation: {log_formation.index[0].date()} -> {log_formation.index[-1].date()} ({len(log_formation)} j)')
print(f'Trading  : {log_trading.index[0].date()} -> {log_trading.index[-1].date()} ({len(log_trading)} j)')

In [ ]:
# re-estimer les hedge ratios sur la période de formation seulement (pas de double ADF)
hedge_formation = {}
for stock in selected:
    _, _, h = engle_granger_test(log_formation[stock], log_formation[ANCHOR])
    hedge_formation[stock] = h

optimal_weights = optimize_basket(
    selected_stocks=selected, log_prices=log_formation,
    hedge_ratios=hedge_formation, allow_short=True, verbose=False
)

print('Poids optimaux:')
for stock, w in optimal_weights.items():
    direction = 'LONG ' if w > 0 else 'SHORT'
    print(f'  {direction}  {stock:8s}  w={w:+.4f}   beta={hedge_formation[stock]:+.4f}')

net_beta = sum(hedge_formation[s] * optimal_weights[s] for s in selected)
print(f'Net Beta = {net_beta:.6f}  (cible = 0)')
print(f'Sum w    = {optimal_weights.sum():.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['steelblue' if w > 0 else 'tomato' for w in optimal_weights.values]
bars = ax.bar(optimal_weights.index, optimal_weights.values, color=colors, edgecolor='black', linewidth=0.7)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Poids du basket market-neutral (Gurobi)', fontsize=12)
ax.set_ylabel('Poids')
for bar, val in zip(bars, optimal_weights.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + (0.01 if val >= 0 else -0.03),
            f'{val:+.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig_weights.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Backtest - mean reversion sur Z-score

Spread = somme pondérée des log-prix. Z-score rolling (fenêtre 60 jours).  
Entrée si |Z| > 2, sortie quand Z revient à 0. Coûts: 10 bps par leg.

In [ ]:
def compute_spread(log_p, weights):
    return (log_p[weights.index] * weights.values).sum(axis=1)


def rolling_zscore(series, window=60):
    mu  = series.rolling(window).mean()
    sig = series.rolling(window).std()
    return (series - mu) / sig


def backtest(log_p, prices, weights, z_entry=2.0, z_exit=0.0,
             lookback=60, tx_cost_bps=10, initial_capital=100_000):
    tc     = tx_cost_bps / 10_000
    spread = compute_spread(log_p, weights)
    zscore = rolling_zscore(spread, lookback).dropna()

    common = log_p.index.intersection(zscore.index)
    log_p  = log_p.loc[common]
    zscore = zscore.loc[common]
    spread = spread.loc[common]

    stocks  = weights.index.tolist()
    n       = len(stocks)
    w_arr   = weights.values                    # numpy array — plus rapide
    lp_arr  = log_p[stocks].values              # (T, n) numpy array
    z_arr   = zscore.values                     # (T,) numpy array
    dates   = log_p.index

    position   = 0
    equity     = initial_capital
    entry_lp   = None
    entry_z    = None
    entry_date = None

    eq_list    = []
    spread_list = spread.values.tolist()
    z_list      = z_arr.tolist()
    pos_list    = []
    trade_log  = []

    for i in range(len(dates)):
        z   = z_list[i]
        clp = lp_arr[i]          # log-prices at time i

        # --- sortie ---
        if position != 0:
            if (position == 1 and z >= z_exit) or (position == -1 and z <= z_exit):
                ret_vec = clp - entry_lp
                gross   = float(w_arr.dot(ret_vec)) * position
                cost    = tc * n
                net     = gross - cost
                equity += net * equity
                trade_log.append({
                    'open_date' : entry_date,
                    'close_date': dates[i],
                    'direction' : 'LONG' if position == 1 else 'SHORT',
                    'entry_z'   : round(entry_z, 3),
                    'exit_z'    : round(z, 3),
                    'gross_ret' : round(gross, 5),
                    'cost'      : round(cost, 5),
                    'net_ret'   : round(net, 5),
                    'pnl_$'     : round(net * (equity / (1 + net)), 2)
                })
                position   = 0
                entry_lp   = None

        # --- entrée ---
        if position == 0:
            if z > z_entry:
                position   = -1
                entry_lp   = clp.copy()
                entry_date = dates[i]
                entry_z    = z
                equity    -= tc * n * equity
            elif z < -z_entry:
                position   = 1
                entry_lp   = clp.copy()
                entry_date = dates[i]
                entry_z    = z
                equity    -= tc * n * equity

        eq_list.append(equity)
        pos_list.append(position)

    portfolio = pd.DataFrame({
        'spread'  : spread_list,
        'zscore'  : z_list,
        'position': pos_list,
        'equity'  : eq_list
    }, index=dates)

    trades = pd.DataFrame(trade_log)
    return portfolio, trades


print('ok')

In [ ]:
LOOKBACK    = 60
Z_ENTRY     = 2.0
Z_EXIT      = 0.0
TX_COST_BPS = 10
CAPITAL     = 100_000

portfolio, trades = backtest(
    log_p=log_trading, prices=prices_trading, weights=optimal_weights,
    z_entry=Z_ENTRY, z_exit=Z_EXIT, lookback=LOOKBACK,
    tx_cost_bps=TX_COST_BPS, initial_capital=CAPITAL
)

print(f'Trades: {len(trades)}')
if len(trades) > 0:
    print(f'Gagnants: {(trades["net_ret"] > 0).sum()} / {len(trades)}')
    trades.head(10)

## 6. Métriques de performance

In [ ]:
def compute_metrics(portfolio, trades, risk_free_annual=0.04, label='Strategy'):
    equity = portfolio['equity']
    daily_ret = equity.pct_change().dropna()
    n_years = len(equity) / 252

    total_ret = (equity.iloc[-1] / equity.iloc[0]) - 1
    ann_ret   = (1 + total_ret) ** (1 / n_years) - 1
    ann_vol   = daily_ret.std() * np.sqrt(252)
    rf_daily  = risk_free_annual / 252
    sharpe    = (daily_ret.mean() - rf_daily) / daily_ret.std() * np.sqrt(252) if daily_ret.std() > 0 else np.nan

    rolling_max = equity.cummax()
    drawdown    = (equity - rolling_max) / rolling_max
    max_dd      = drawdown.min()
    calmar      = ann_ret / abs(max_dd) if max_dd != 0 else np.nan

    if len(trades) > 0:
        win_rate = (trades['net_ret'] > 0).mean()
        avg_win  = trades.loc[trades['net_ret'] > 0, 'pnl_$'].mean()
        avg_loss = trades.loc[trades['net_ret'] <= 0, 'pnl_$'].mean()
    else:
        win_rate = avg_win = avg_loss = np.nan

    return {
        'Strategy'            : label,
        'Total Return (%)'    : round(total_ret * 100, 2),
        'Ann. Return (%)'     : round(ann_ret * 100, 2),
        'Ann. Volatility (%)' : round(ann_vol * 100, 2),
        'Sharpe Ratio'        : round(sharpe, 3),
        'Max Drawdown (%)'    : round(max_dd * 100, 2),
        'Calmar Ratio'        : round(calmar, 3),
        '# Trades'            : len(trades),
        'Win Rate (%)'        : round(win_rate * 100, 2) if not np.isnan(win_rate) else 'N/A',
        'Avg Win ($)'         : round(avg_win, 2) if not np.isnan(avg_win) else 'N/A',
        'Avg Loss ($)'        : round(avg_loss, 2) if not np.isnan(avg_loss) else 'N/A',
    }


metrics_strat = compute_metrics(portfolio, trades, label='OTT Basket (Gurobi)')
for k, v in metrics_strat.items():
    print(f'  {k:<25s}: {v}')

## 7. Benchmark et comparaison

In [ ]:
def benchmark_equal_weight(log_p, stocks, initial_capital=100_000, tx_cost_bps=10):
    tc = tx_cost_bps / 10_000
    n = len(stocks)
    ew = (1 / n) * np.ones(n)
    log_rets = log_p[stocks].diff().dropna()
    portfolio_ret = (log_rets * ew).sum(axis=1)
    equity = initial_capital * (1 - tc * n)
    equity_series = equity * np.exp(portfolio_ret.cumsum())
    equity_series = pd.concat([
        pd.Series([equity], index=[log_p.index[0]]),
        equity_series
    ])
    return equity_series


bm_equity = benchmark_equal_weight(log_p=log_trading, stocks=selected,
                                    initial_capital=CAPITAL, tx_cost_bps=TX_COST_BPS)

bm_ret    = bm_equity.pct_change().dropna()
n_years   = len(bm_equity) / 252
bm_total  = (bm_equity.iloc[-1] / bm_equity.iloc[0]) - 1
bm_ann    = (1 + bm_total) ** (1 / n_years) - 1
bm_vol    = bm_ret.std() * np.sqrt(252)
bm_sharpe = (bm_ret.mean() - 0.04/252) / bm_ret.std() * np.sqrt(252)
bm_dd     = ((bm_equity - bm_equity.cummax()) / bm_equity.cummax()).min()
bm_calmar = bm_ann / abs(bm_dd)

metrics_bm = {
    'Strategy'            : 'Equal-Weight Benchmark',
    'Total Return (%)'    : round(bm_total * 100, 2),
    'Ann. Return (%)'     : round(bm_ann * 100, 2),
    'Ann. Volatility (%)' : round(bm_vol * 100, 2),
    'Sharpe Ratio'        : round(bm_sharpe, 3),
    'Max Drawdown (%)'    : round(bm_dd * 100, 2),
    'Calmar Ratio'        : round(bm_calmar, 3),
    '# Trades'            : 1,
}

for k, v in metrics_bm.items():
    print(f'  {k:<25s}: {v}')

In [ ]:
keys = ['Total Return (%)', 'Ann. Return (%)', 'Ann. Volatility (%)',
        'Sharpe Ratio', 'Max Drawdown (%)', 'Calmar Ratio', '# Trades']

comparison = pd.DataFrame([
    {k: metrics_strat.get(k, 'N/A') for k in keys},
    {k: metrics_bm.get(k, 'N/A') for k in keys}
], index=['OTT Basket (Gurobi)', 'Equal-Weight Benchmark'])

print(comparison.to_string())
comparison

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=False)

ax = axes[0]
ax.plot(portfolio.index, portfolio['equity'], label='OTT Basket (Gurobi)', linewidth=2)
ax.plot(bm_equity.index, bm_equity.values, label='Equal-Weight Benchmark',
        linewidth=1.5, linestyle='--')
ax.axhline(CAPITAL, color='grey', linestyle=':', linewidth=1)
ax.set_title('Courbe de valeur du portfolio (2023-2026)', fontsize=12)
ax.set_ylabel('Valeur ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()

ax = axes[1]
strat_eq = portfolio['equity']
strat_dd = (strat_eq - strat_eq.cummax()) / strat_eq.cummax() * 100
bm_dd_ts = (bm_equity - bm_equity.cummax()) / bm_equity.cummax() * 100
ax.fill_between(portfolio.index, strat_dd, 0, alpha=0.3, label='OTT Drawdown')
ax.plot(bm_equity.index, bm_dd_ts, linewidth=1, linestyle='--', label='Benchmark')
ax.set_title('Drawdown (%)', fontsize=12)
ax.set_ylabel('Drawdown (%)')
ax.legend()

ax = axes[2]
ax.plot(portfolio.index, portfolio['zscore'], linewidth=1, label='Z-Score')
ax.axhline(Z_ENTRY,  color='red',   linestyle='--', linewidth=1, label=f'+{Z_ENTRY}σ')
ax.axhline(-Z_ENTRY, color='green', linestyle='--', linewidth=1, label=f'-{Z_ENTRY}σ')
ax.axhline(0, color='black', linestyle=':', linewidth=0.7)
if len(trades) > 0:
    long_entries  = trades[trades['direction']=='LONG']['open_date']
    short_entries = trades[trades['direction']=='SHORT']['open_date']
    ax.scatter(long_entries,  portfolio.loc[long_entries, 'zscore'],
               marker='^', color='green', zorder=5, label='Long', s=60)
    ax.scatter(short_entries, portfolio.loc[short_entries, 'zscore'],
               marker='v', color='red', zorder=5, label='Short', s=60)
ax.set_title('Z-Score avec signaux', fontsize=12)
ax.set_ylabel('Z-Score')
ax.legend(fontsize=8, ncol=3)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('figures/fig_results.png', dpi=150, bbox_inches='tight')
plt.show()

if len(trades) > 0:
    trades['close_date'] = pd.to_datetime(trades['close_date'])
    monthly_pnl = trades.set_index('close_date')['pnl_$'].resample('ME').sum()

    fig, ax = plt.subplots(figsize=(12, 4))
    colors = ['steelblue' if p >= 0 else 'tomato' for p in monthly_pnl]
    ax.bar(monthly_pnl.index, monthly_pnl.values, color=colors, width=20,
           edgecolor='black', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title('PnL mensuel - OTT basket', fontsize=12)
    ax.set_ylabel('PnL ($)')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('figures/fig_monthly_pnl.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('pas de trades')

## 8. Sensibilité au seuil Z

On fait tourner le backtest sur plusieurs seuils pour tester la robustesse.

In [ ]:
thresholds = [1.5, 1.75, 2.0, 2.25, 2.5]
sensitivity = []

for z in thresholds:
    p, t = backtest(
        log_p=log_trading, prices=prices_trading, weights=optimal_weights,
        z_entry=z, z_exit=Z_EXIT, lookback=LOOKBACK,
        tx_cost_bps=TX_COST_BPS, initial_capital=CAPITAL
    )
    m = compute_metrics(p, t, label=f'z={z}')
    sensitivity.append(m)

sens_df = pd.DataFrame(sensitivity)[[
    'Strategy', 'Ann. Return (%)', 'Sharpe Ratio',
    'Max Drawdown (%)', 'Calmar Ratio', '# Trades', 'Win Rate (%)'
]].set_index('Strategy')

sens_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics_to_plot = ['Ann. Return (%)', 'Sharpe Ratio', '# Trades']

for ax, metric in zip(axes, metrics_to_plot):
    vals = [s[metric] for s in sensitivity]
    ax.bar(thresholds, vals, edgecolor='black', linewidth=0.6, width=0.18)
    ax.set_title(metric, fontsize=11)
    ax.set_xlabel('Z-Entry')
    ax.axhline(0, color='black', linewidth=0.7)

plt.suptitle('Sensibilite au seuil Z', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('figures/fig_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Bilan

Le basket est market-neutral (beta net = 0), ce qui élimine l'exposition aux futures café.  
La stratégie exploite la mean-reversion du spread cointégré.

Limite principale: les hedge ratios sont estimés sur la formation — risque de rupture out-of-sample.